# Task 1: News Topic Classifier Using BERT
AG News + BERT fine-tuning using Hugging Face Transformers

In [ ]:
pip install transformers datasets evaluate scikit-learn torch gradio -q

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
dataset = load_dataset('ag_news')
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)
tokenized = dataset.map(tokenize, batched=True)
tokenized = tokenized.rename_column('label','labels')


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4)
args = TrainingArguments(output_dir='./results', num_train_epochs=1, per_device_train_batch_size=16)
trainer = Trainer(model=model, args=args, train_dataset=tokenized['train'].select(range(5000)), eval_dataset=tokenized['test'].select(range(1000)))
trainer.train()


In [ ]:
pred = trainer.predict(tokenized['test'].select(range(1000)))
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
y_pred = np.argmax(pred.predictions, axis=1)
y_true = pred.label_ids
print('Accuracy:', accuracy_score(y_true,y_pred))
print('F1:', f1_score(y_true,y_pred, average='weighted'))
